# Processed Descriptor Visualization and QA

This notebook inspects the **current processed sample bundles** under `data/processed/samples/...`.
It is meant to answer practical workflow questions:

1. Are the processed descriptor maps populated and visually sensible?
2. Do the audit reports indicate clean category coverage and descriptor-policy consistency?
3. Does a selected processed sample reflect the expected thermal, geometric, and cross-modal outputs for the current workflow?


In [1]:
from pathlib import Path
import csv
import io
import json
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    pd = None
    PANDAS_AVAILABLE = False

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    Image = None
    PIL_AVAILABLE = False

try:
    from IPython import get_ipython
    from IPython.display import Image as IPythonImage, display
except ImportError:
    get_ipython = None
    IPythonImage = None
    display = None


def configure_notebook_backend():
    ip = get_ipython() if get_ipython is not None else None
    if ip is None:
        return False
    try:
        ip.run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
    return True


IN_NOTEBOOK = configure_notebook_backend()


def render_figure(fig):
    fig.tight_layout()
    if IN_NOTEBOOK and display is not None and IPythonImage is not None:
        buffer = io.BytesIO()
        fig.savefig(buffer, format='png', bbox_inches='tight', dpi=160)
        buffer.seek(0)
        display(IPythonImage(data=buffer.getvalue()))
        buffer.close()
    else:
        fig.canvas.draw()
    plt.close(fig)

print('Pandas available   :', PANDAS_AVAILABLE)
print('Pillow available   :', PIL_AVAILABLE)
print('Matplotlib backend :', plt.get_backend())

if not PIL_AVAILABLE:
    raise RuntimeError('Pillow is required for the processed descriptor QA notebook.')


Pandas available   : True
Pillow available   : True
Matplotlib backend : module://matplotlib_inline.backend_inline


In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'processed' / 'samples').exists() and (candidate / 'src' / 'notebooks').exists():
            return candidate
    raise RuntimeError(f'Could not locate project root from {start}')


PROJECT_ROOT = find_project_root()
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
SAMPLES_ROOT = PROCESSED_ROOT / 'samples'
INDEX_CSV = PROCESSED_ROOT / 'manifests' / 'dataset_index.csv'
DATA_AUDIT_JSON = PROCESSED_ROOT / 'reports' / 'data_audit.json'
POLICY_AUDIT_JSON = PROCESSED_ROOT / 'reports' / 'descriptor_policy_audit.json'
POLICY_AUDIT_CSV = PROCESSED_ROOT / 'reports' / 'descriptor_policy_audit.csv'

for required in [SAMPLES_ROOT, INDEX_CSV, DATA_AUDIT_JSON, POLICY_AUDIT_JSON, POLICY_AUDIT_CSV]:
    assert required.exists(), f'Missing required processed artifact: {required}'

rows = list(csv.DictReader(INDEX_CSV.open(newline='', encoding='utf-8')))
sample_dirs = sorted([path for path in SAMPLES_ROOT.iterdir() if path.is_dir()])

print('Project root       :', PROJECT_ROOT)
print('Processed root     :', PROCESSED_ROOT)
print('Processed samples  :', len(sample_dirs))
print('Indexed rows       :', len(rows))


Project root       : /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026
Processed root     : /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/processed
Processed samples  : 2035
Indexed rows       : 2035


## Audit Snapshot

These reports summarize whether the processed dataset is structurally complete and whether category policies are reflected consistently across processed bundles.


In [3]:
data_audit = json.loads(DATA_AUDIT_JSON.read_text(encoding='utf-8'))
policy_audit_json = json.loads(POLICY_AUDIT_JSON.read_text(encoding='utf-8'))
policy_rows = list(csv.DictReader(POLICY_AUDIT_CSV.open(newline='', encoding='utf-8')))

print('Data audit record_count :', data_audit['record_count'])
print('Data audit issue_count  :', data_audit['issue_count'])
print('Policy audit rows       :', len(policy_rows))
print('Policy names            :', sorted({row['descriptor_policy'] for row in policy_rows}))

per_category_rows = []
for category_name in data_audit['categories']:
    per_category_rows.append({'category': category_name, **data_audit['per_category'][category_name]})

if PANDAS_AVAILABLE:
    display(pd.DataFrame(per_category_rows))
    display(pd.DataFrame(policy_rows))
else:
    for row in per_category_rows[:5]:
        print(row)
    for row in policy_rows[:5]:
        print(row)


Data audit record_count : 2035
Data audit issue_count  : 0
Policy audit rows       : 15
Policy names            : ['default_geometry', 'residual_reference']


,category,anomalous_test,pointcloud_gt_available,rgb_gt_available,test,train
0,button_cell,31,15,26,41,90
1,capsule,48,33,41,58,64
2,cotton,40,40,8,50,78
3,cube,41,33,33,51,110
4,flat_pad,30,23,30,40,90
5,light,36,21,12,46,110
6,nut,29,14,29,39,118
7,piggy,30,18,29,40,110
8,plastic_cylinder,28,21,17,38,90
9,screen,32,32,29,42,69


,category,descriptor_policy,sample_count,train_count,test_count,anomalous_count,depth_reference_mean_present,depth_reference_std_present,normal_reference_mean_present,normal_reference_std_present,pc_curvature_semantics,pc_roughness_semantics,pc_normal_deviation_semantics
0,button_cell,default_geometry,131,90,41,31,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
1,capsule,default_geometry,122,64,58,48,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
2,cotton,default_geometry,128,78,50,40,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
3,cube,default_geometry,161,110,51,41,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
4,flat_pad,default_geometry,130,90,40,30,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
5,light,default_geometry,156,110,46,36,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
6,nut,default_geometry,157,118,39,29,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
7,piggy,default_geometry,150,110,40,30,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
8,plastic_cylinder,default_geometry,128,90,38,28,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
9,screen,default_geometry,111,69,42,32,1,0,1,0,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized


In [4]:
policy_counter = Counter(row['descriptor_policy'] for row in policy_rows)
semantics_rows = []
for row in policy_rows:
    semantics_rows.append({
        'category': row['category'],
        'descriptor_policy': row['descriptor_policy'],
        'pc_curvature_semantics': row['pc_curvature_semantics'],
        'pc_roughness_semantics': row['pc_roughness_semantics'],
        'pc_normal_deviation_semantics': row['pc_normal_deviation_semantics'],
    })

print('Descriptor policy counts:', dict(policy_counter))

if PANDAS_AVAILABLE:
    display(pd.DataFrame(semantics_rows))
else:
    for row in semantics_rows:
        print(row)


Descriptor policy counts: {'default_geometry': 13, 'residual_reference': 2}


,category,descriptor_policy,pc_curvature_semantics,pc_roughness_semantics,pc_normal_deviation_semantics
0,button_cell,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
1,capsule,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
2,cotton,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
3,cube,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
4,flat_pad,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
5,light,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
6,nut,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
7,piggy,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
8,plastic_cylinder,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized
9,screen,default_geometry,curvature_residual_normalized,roughness_residual_normalized,normal_deviation_normalized


## Sample Bundle Selection

Pick a processed sample bundle for visual QA. The default points to a geometry-sensitive anomalous sample so the descriptor-policy fields are easy to inspect.


In [5]:
SAMPLE_NAME = 'screw__test__broken__003'
CATEGORY_FILTER = None
REQUIRE_ANOMALOUS = True

print({
    'SAMPLE_NAME': SAMPLE_NAME,
    'CATEGORY_FILTER': CATEGORY_FILTER,
    'REQUIRE_ANOMALOUS': REQUIRE_ANOMALOUS,
})


{'SAMPLE_NAME': 'screw__test__broken__003', 'CATEGORY_FILTER': None, 'REQUIRE_ANOMALOUS': True}


In [6]:
def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def select_sample_dir(sample_name=None, category_filter=None, require_anomalous=True):
    if sample_name is not None:
        selected = SAMPLES_ROOT / sample_name
        if not selected.exists():
            raise RuntimeError(f'Requested sample bundle not found: {selected}')
        return selected

    candidates = []
    for sample_dir in sample_dirs:
        meta = load_json(sample_dir / 'meta.json')
        if category_filter and meta['category'] != category_filter:
            continue
        is_anomalous = meta['defect_label'] != 'good'
        if require_anomalous and not is_anomalous:
            continue
        candidates.append(sample_dir)

    if not candidates:
        raise RuntimeError('No processed sample bundle matched the requested filters.')
    return candidates[0]


selected_dir = select_sample_dir(SAMPLE_NAME, CATEGORY_FILTER, REQUIRE_ANOMALOUS)
selected_meta = load_json(selected_dir / 'meta.json')
selected_global = load_json(selected_dir / 'global.json')

print('Selected sample dir:', selected_dir)
print('Selected meta      :', selected_meta)
print('Selected global    :')
print(json.dumps(selected_global, indent=2, sort_keys=True))


Selected sample dir: /home/eddy/Downloads/sirharginger/MulSenDiff-X-IEEE-IES-2026/MulSenDiff-X-IEEE-IES-2026/data/processed/samples/screw__test__broken__003
Selected meta      : {'category': 'screw', 'defect_label': 'broken', 'descriptor_policy': 'residual_reference', 'gt_mask_path': 'data/processed/samples/screw__test__broken__003/gt_mask.png', 'ir_source_path': 'data/raw/MulSen_AD/screw/Infrared/test/broken/3.png', 'pc_source_path': 'data/raw/MulSen_AD/screw/Pointcloud/test/broken/3.stl', 'pointcloud_channel_semantics': {'pc_curvature': 'zero_dummy_channel', 'pc_density': 'density_normalized', 'pc_depth': 'depth_residual_normalized', 'pc_normal_deviation': 'normal_reference_residual_magnitude_normalized', 'pc_roughness': 'depth_reference_residual_normalized'}, 'pointcloud_global_semantics': {'pc_max_roughness': 'max_depth_reference_residual', 'pc_mean_curvature': 'mean_depth_reference_residual', 'pc_p95_normal_deviation': 'p95_normal_reference_residual'}, 'rgb_source_path': 'data/raw

In [7]:
selected_policy = next((row for row in policy_rows if row['category'] == selected_meta['category']), None)
if selected_policy is None:
    raise RuntimeError(f'Could not find a policy audit row for category {selected_meta["category"]}')

policy_view = {
    'category': selected_policy['category'],
    'descriptor_policy': selected_policy['descriptor_policy'],
    'sample_count': selected_policy['sample_count'],
    'anomalous_count': selected_policy['anomalous_count'],
    'pc_curvature_semantics': selected_policy['pc_curvature_semantics'],
    'pc_roughness_semantics': selected_policy['pc_roughness_semantics'],
    'pc_normal_deviation_semantics': selected_policy['pc_normal_deviation_semantics'],
}

print('Policy summary for selected category:')
print(json.dumps(policy_view, indent=2, sort_keys=True))
print('\nBundle semantics from meta.json:')
print(json.dumps(selected_meta.get('pointcloud_channel_semantics', {}), indent=2, sort_keys=True))


Policy summary for selected category:
{
  "anomalous_count": "31",
  "category": "screw",
  "descriptor_policy": "residual_reference",
  "pc_curvature_semantics": "zero_dummy_channel",
  "pc_normal_deviation_semantics": "normal_reference_residual_magnitude_normalized",
  "pc_roughness_semantics": "depth_reference_residual_normalized",
  "sample_count": "131"
}

Bundle semantics from meta.json:
{
  "pc_curvature": "zero_dummy_channel",
  "pc_density": "density_normalized",
  "pc_depth": "depth_residual_normalized",
  "pc_normal_deviation": "normal_reference_residual_magnitude_normalized",
  "pc_roughness": "depth_reference_residual_normalized"
}


## Descriptor Map Statistics

Quick numeric checks help answer whether the processed maps are populated and nontrivial before visual inspection.


In [8]:
def image_stats(path):
    array = np.asarray(Image.open(path).convert('L'), dtype=np.float32)
    return {
        'artifact': path.name,
        'min': float(array.min()),
        'max': float(array.max()),
        'mean': float(array.mean()),
        'std': float(array.std()),
        'nonzero_ratio': float((array > 0).mean()),
    }


stat_paths = [
    path for path in sorted(selected_dir.glob('*.png'))
    if path.name not in {'rgb.png', 'gt_mask.png', 'cross_agreement_preview.png'}
]
stat_rows = [image_stats(path) for path in stat_paths]

if PANDAS_AVAILABLE:
    display(pd.DataFrame(stat_rows))
else:
    for row in stat_rows:
        print(row)


,artifact,min,max,mean,std,nonzero_ratio
0,cross_agreement.png,0.0,0.0,0.000000,0.000000,0.000000
1,cross_geo_support.png,0.0,0.0,0.000000,0.000000,0.000000
2,cross_inconsistency.png,0.0,141.0,1.678680,7.335334,0.458389
3,cross_ir_support.png,0.0,141.0,1.678680,7.335334,0.458389
4,ir_gradient.png,0.0,141.0,1.678680,7.335334,0.458389
5,ir_hotspot.png,0.0,0.0,0.000000,0.000000,0.000000
6,ir_normalized.png,0.0,255.0,232.656113,31.237438,0.998505
7,ir_variance.png,0.0,80.0,0.342026,3.583785,0.014542
8,pc_curvature.png,0.0,0.0,0.000000,0.000000,0.000000
9,pc_density.png,0.0,212.0,6.957855,12.644666,0.431168


## Visual Descriptor Panel

The panel below shows the raw RGB view together with the processed thermal, geometric, and cross-modal outputs from the same sample bundle.


In [9]:
DISPLAY_ORDER = [
    'rgb.png',
    'gt_mask.png',
    'ir_normalized.png',
    'ir_hotspot.png',
    'ir_gradient.png',
    'ir_variance.png',
    'pc_depth.png',
    'pc_density.png',
    'pc_curvature.png',
    'pc_roughness.png',
    'pc_normal_deviation.png',
    'cross_ir_support.png',
    'cross_geo_support.png',
    'cross_agreement.png',
    'cross_inconsistency.png',
]


def load_display_array(path):
    if path.name == 'rgb.png':
        return np.asarray(Image.open(path).convert('RGB')), None
    if path.name == 'gt_mask.png':
        return np.asarray(Image.open(path).convert('L')), 'gray'
    if path.name.startswith('pc_'):
        return np.asarray(Image.open(path).convert('L')), 'viridis'
    if path.name.startswith('cross_'):
        return np.asarray(Image.open(path).convert('L')), 'magma'
    return np.asarray(Image.open(path).convert('L')), 'inferno'


present_paths = [selected_dir / name for name in DISPLAY_ORDER if (selected_dir / name).exists()]
columns = 4
rows_needed = (len(present_paths) + columns - 1) // columns
fig, axes = plt.subplots(rows_needed, columns, figsize=(16, 4 * rows_needed))
axes = np.atleast_1d(axes).reshape(rows_needed, columns)

for ax, path in zip(axes.flatten(), present_paths):
    array, cmap = load_display_array(path)
    if array.ndim == 3:
        ax.imshow(array)
    else:
        ax.imshow(array, cmap=cmap)
    ax.set_title(path.stem)
    ax.set_xticks([])
    ax.set_yticks([])

for ax in axes.flatten()[len(present_paths):]:
    ax.axis('off')

render_figure(fig)


## QA Interpretation Notes

- Use the audit summary cells to confirm that the processed dataset is structurally complete and policy-consistent.
- Use the global descriptor summary to confirm that cross-modal and thermal statistics are being written into each sample bundle.
- Use the per-image statistics and descriptor panel to judge whether the maps are non-empty, spatially selective, and visually plausible enough for current preprocessing QA and paper figure preparation.
- If a future notebook is needed for detector outputs, it should be created against the final eval artifacts rather than restoring the removed legacy manifest workflow.
